In [1]:
%cd ..

/Users/philipphager/Documents/GitHub/clix


In [2]:
import pandas as pd
import altair as alt

from pathlib import Path

In [3]:
result_dir = Path("clax-results/4-baidu-ultr-features/")

In [4]:
click_df = pd.concat([pd.read_csv(f) for f in list(result_dir.glob("*/test_[!r]*.csv"))], ignore_index=True)
click_df.head()

,model,test_loss,test_ll,test_ppl,test_cond_ppl,train_time_s
0,DBN,0.209511,-0.213246,1.228622,1.217681,684.327921
1,SDBN,0.222927,-0.227369,1.236993,1.236136,496.656547
2,DCM,0.223685,-0.228164,1.238234,1.237346,556.563549
3,CCM,0.209976,-0.213772,1.229248,1.218345,694.837676
4,GCTR,0.288909,-0.292676,1.328223,1.328223,321.792693


In [5]:
rel_df = pd.concat([pd.read_csv(f) for f in list(result_dir.glob("*/test_rel_[!r]*.csv"))], ignore_index=True)
rel_df.head()

,model,test_dcg@10,test_dcg@5,test_dcg@3,test_dcg@1,test_mrr@10
0,DCM,7.623711,4.985905,3.641592,1.735290,0.626570
1,DBN,7.776213,5.138557,3.762371,1.804725,0.614429
2,DCTR,7.538476,4.944446,3.603718,1.721976,0.622865
3,CCM,7.658573,5.020501,3.646527,1.729707,0.621645
4,PBM,7.705985,5.012168,3.626111,1.702649,0.621538


In [6]:
model2color = {
    "PBM": "#3182bd",
    "UBM": "#6baed6",
    "DBN": "#31a354",
    "SDBN": "#74c476",
    "CM": "#fd8d3c",
    "CCM": "#fdae6b",
    "DCM": "#fdd0a2",
    "DCTR": "#969696",
    "RCTR": "#969696",
    "GCTR": "#bdbdbd",
}

In [8]:
def plot_metric(df, metric, domain, y_title, ascending=False, scheme="category20c"):
    df = df[df[metric].notna()]
    model_order = (
        df
        .groupby("model")[metric]
        .mean()
        .sort_values(ascending=ascending)
        .index
    )
    

    source = df[df.model.isin(model_order)]
    source["color"] = df.model.map(model2color)
    base = alt.Chart(source, width=225, height=200)

    x = alt.X("model", title="", sort=model_order).axis(labelAngle=45)
    
    bars = base.mark_bar().encode(
        x=x,
        y=alt.Y(f"mean({metric})", title=y_title).scale(domain=domain, clamp=True),
        color=alt.Color("color").scale(None)
    )
    errors = base.mark_errorbar(extent="ci", thickness=3).encode(
        x=x,
        y=alt.Y(metric, title=y_title),
    )
    
    return bars + errors

ppl = plot_metric(click_df, metric="test_ppl", domain=(1.2, 1.34), y_title="PPL", ascending=True)
cond_ppl = plot_metric(click_df, metric="test_cond_ppl", domain=(1.2, 1.34), y_title="Conditional PPL", ascending=True)
dcg = plot_metric(rel_df, metric="test_dcg@10", domain=(6, 9), y_title="DCG@10")

chart = ppl | cond_ppl | dcg
chart = chart.configure_concat(spacing=0).configure_scale(bandWithNestedOffsetPaddingInner=0.16)
chart

alt.HConcatChart(...)